# CS-781 Phase 2 — Legal NER pipeline (notebook E)

**What it does:** fine-tunes YOUR Phase-1 models for legal named-entity recognition and writes Codabench submissions.
The pipeline (data, split, tokenisation with exact character offsets, windows, decoding, metrics, files) is written;
**your part is one cell: N6 `ner_methods.py`** (the NER head, the fine-tuning methods incl. LORMA, and knowledge distillation).

**Kaggle settings:** Accelerator **GPU T4 ×2** (for the CPU smoke test: None) · Internet **On** · secret `Capstone_Phase_1` on.

**Inputs (Add Input) — your Phase-1 checkpoints + data prep:**
| Input | Provides |
|---|---|
| `notebook857159535b` (data prep) | your tokenizer (`bpe_tok.py`, `bpe.json`) |
| notebook D (`phase1-continue`) | `continued/best_continued.pt` (decoder, val loss 2.656) |
| `Phase1_Training1` (latest = session 2) | `runs/encdec/best.pt`, `runs/swa/best.pt` |
| dataset **`prefix`** (you upload `runs/prefix/best.pt` from `Phase1_Training1` version 1, like `decoder`) | `prefix/best.pt` |
| `phase1-train-extra` (session 3) | `runs/diffusion/best.pt`, `runs/moe/best.pt` |

**How to use**
1. **CPU session first:** run N1 → N8. N8 checks the data + checkpoints and runs a **smoke test** of your code (tiny random models, a few documents, ~1–2 min per experiment). Fix your code until every experiment prints `exit code 0`.
2. **GPU run:** switch to GPU T4 ×2, choose `SESSION_EXPERIMENTS` in N3 → **Save & Run All**. Two experiments train at the same time (one per GPU); a distillation student waits for its teacher.
3. After the run: N10 (scores table), N11 (error analysis), N12 (zip check) → upload `ner_runs/<experiment>/submission_<experiment>.zip` files to Codabench.

## N1. Login + download the Phase 2 data

In [ ]:
import os, shutil, json
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("Capstone_Phase_1")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
os.makedirs("/kaggle/working/p2data", exist_ok=True)
for path in ["Train/fold1-text.json", "Train/fold1-gold.json", "Train/fold2-text.json", "Train/fold2-gold.json",
             "Train/labels.txt", "Test/fold3-text.json"]:
    src = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_2_LNER/{path}", repo_type="dataset", token=HF_TOKEN)
    shutil.copy(src, f"/kaggle/working/p2data/{os.path.basename(path)}")
print(sorted(os.listdir("/kaggle/working/p2data")))

## N2. GPU check (0 GPUs is fine for the CPU smoke test)

In [ ]:
import torch
print("GPU count:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## N3. Configuration — experiments and settings (read the comments: what each value does and the trade-offs)

In [ ]:
import json
# =====================================================================================================
#  N3 — CONFIGURATION. Values marked YOUR CHOICE are yours (course rule §17.5); the comments explain each one.
#  Write down why you chose them: the report must compare methods and justify settings.
# =====================================================================================================
#
#  PHASE 2 RULES THIS CONFIG MUST SATISFY (§8)
#  * all FOUR required Phase-1 architectures used: decoder, prefix (hybrid), encdec, swa (efficient)
#  * the best four submissions come from DIFFERENT Phase-1 models
#  * at least 3 different fine-tuning / parameter-efficient techniques — LORMA is COMPULSORY
#  * at least 1 knowledge-distillation experiment (bigger teacher -> smaller student) — COMPULSORY
#  * ≥ 6 submissions in total, each a different METHOD (not just other learning rates / ranks)
#
#  WHAT YOUR DATA SAID (exploration notebook) — use it when choosing
#  * 70 training documents, 35 test; median 5,300 tokens per document -> every document needs several windows
#  * plain BPE could reach only 69.7% of entities exactly; the punctuation split used here reaches 99.55%
#  * rare labels: A.COUNSEL, R.COUNSEL, WIT ≈ 2% each; WIT occurs in only 12 documents
#  * one experiment ≈ 650 training windows per epoch ≈ about 1 minute per epoch for a 55M model on one T4

CONFIG = {
    "seed": 42,

    # ---------------- validation split (same for EVERY experiment, so comparisons are fair) ----------------
    "n_val_docs": 14,        # documents held out for validation (20% of 70). More = more reliable F1 estimate,
                             # fewer = more training data. Chosen automatically so each label has ~val_fraction
                             # of its entities in validation (WIT included). N8 prints the result.
    "val_fraction": 0.2,

    # ---------------- where each Phase-1 model's checkpoint is (searched under /kaggle/input) ----------------
    "checkpoints": {
        "decoder":   "continued/best_continued.pt",   # notebook D: decoder after continued training (val loss 2.656)
        "prefix":    "prefix/best.pt",                # the dataset you upload (Phase1_Training1 version 1)
        "encdec":    "encdec/best.pt",                # Phase1_Training1 latest version (session 2)
        "swa":       "swa/best.pt",
        "diffusion": "diffusion/best.pt",             # phase1-train-extra (session 3)
        "moe":       "moe/best.pt",
    },

    # ---------------- defaults for every experiment (an experiment below can override any of them) ----------------
    "defaults": {
        "window": 1024,      # tokens per window (automatically capped at what the model accepts: encdec 768).
                             # Longer = more context per prediction, more memory.
        "stride": 768,       # the next window starts this many tokens later -> 256 tokens of overlap.
                             # Each token's prediction is taken from the window where it is most central.
                             # Smaller stride = more overlap = more context at window edges, more compute.
        "context": "plain",  # "plain" or "echo". Causal models (decoder, swa, moe) only see tokens to the LEFT;
                             # but labels often depend on what comes AFTER ("STATE OF HARYANA ..Respondent").
                             # "echo" feeds each window twice and reads the second copy, so every token has seen
                             # the whole window (Springer et al. 2024). Halves the window; doubles compute.
                             # Prefix/diffusion/encdec are already bidirectional here -> "plain".
        "epochs": 10,        # passes over the training windows; the best epoch (validation F1) is kept.
        "patience": 3,       # stop early after this many epochs without a better validation F1.
        "lr": 5e-5,          # YOUR CHOICE. Typical: full fine-tuning 2e-5..1e-4 (BERT used 2e-5..5e-5);
                             # LoRA/LoRMA/BitFit train few parameters and usually need 1e-4..1e-3.
        "batch_size": 8,     # windows per step (8 x 1024 tokens fits a T4; lower it if you run out of memory)
        "warmup_frac": 0.1,  # learning rate rises over the first 10% of steps, then falls linearly to 0
        "weight_decay": 0.01,
        "grad_clip": 1.0,
        "head_dropout": 0.1, # dropout before the NER head (your TokenClassifier)
        "class_weights": "none",   # "none" or "inv_sqrt". Macro-F1 weighs rare labels (WIT, counsel) as much as
                                   # DATE; "inv_sqrt" makes mistakes on rare tags cost more (weight ∝ 1/sqrt(count)).
                                   # Usually raises rare-label recall, may lower precision.
        "decode": "lenient", # "lenient": an I-X tag without a preceding B-X still starts an entity;
                             # "strict": it is ignored. A span-decoding choice the brief asks you to investigate.
        "align": "token",    # "token": every token gets a tag (a predicted entity may start/end INSIDE a word,
                             #          e.g. 'ha Sinha' for "Shantha Sinha").
                             # "piece": only the first token of each piece (a run of letters/digits, or one
                             #          punctuation mark) is tagged and the whole piece takes that tag, so spans
                             #          always start/end on piece boundaries (the usual BERT-NER convention,
                             #          Devlin et al. 2019). A character-to-token alignment choice to compare.
        "method_cfg": {},    # settings passed to your method (e.g. {"r": 8, "alpha": 16})
        "moe_aux_coef": 0.0, # LexMoE only: keep the expert load-balancing loss during fine-tuning (e.g. 0.01)
        "train_on_all": False,     # True = train on all 70 documents (no validation; fixed epochs) — for a final model
        "teacher": None,           # distillation: name of the (finished) teacher experiment
        "kd": {"temperature": 2.0, "alpha": 0.5},   # your kd_loss settings (see N6, K1-K2)
        "student_layers": None,    # distillation: keep only the first k layers of the student model (N6, K3)
    },

    # ---------------- the experiments — YOUR CHOICE (this is a starting plan that meets every rule) ----------------
    # Each entry: "name": {settings that differ from defaults}. "arch" picks the Phase-1 model, "method" picks
    # your function from METHODS in N6.
    "experiments": {
        "prefix_full":    {"arch": "prefix", "method": "full"},                           # hybrid, full fine-tuning
        "encdec_lorma":   {"arch": "encdec", "method": "lorma", "method_cfg": {"r": 8, "alpha": 16},
                           "lr": 5e-4},                                                    # encoder-decoder, LORMA (compulsory)
        "decoder_lora":   {"arch": "decoder", "method": "lora", "method_cfg": {"r": 8, "alpha": 16},
                           "lr": 5e-4},                                                    # decoder-only, LoRA
        "swa_bitfit":     {"arch": "swa", "method": "bitfit", "lr": 1e-3},                 # efficient, BitFit
        "diffusion_full": {"arch": "diffusion", "method": "full"},                         # extra: bidirectional diffusion LM
        "moe_full":       {"arch": "moe", "method": "full"},                               # teacher for distillation (142M)
        "kd_moe_to_decoder4": {"arch": "decoder", "method": "full", "teacher": "moe_full",
                               "student_layers": 4},                                       # distillation (compulsory)
    },
}

# ---------------- which experiments THIS session runs (GPU), and which the CPU smoke test runs ----------------
SESSION_EXPERIMENTS = ["prefix_full", "encdec_lorma", "decoder_lora", "swa_bitfit",
                       "moe_full", "kd_moe_to_decoder4", "diffusion_full"]
SMOKE_EXPERIMENTS = ["prefix_full", "encdec_lorma", "decoder_lora", "swa_bitfit", "moe_full", "kd_moe_to_decoder4"]
#   ^ the smoke test needs only your code (no checkpoints). Teachers must come before their students.

json.dump(CONFIG, open("/kaggle/working/run_config_ner.json", "w"), indent=1)
for name, e in CONFIG["experiments"].items():
    assert e["arch"] in CONFIG["checkpoints"], f"{name}: unknown arch {e['arch']}"
    t = e.get("teacher")
    assert not t or t in CONFIG["experiments"], f"{name}: teacher {t} is not an experiment"
n = len(SESSION_EXPERIMENTS)
print(f"{len(CONFIG['experiments'])} experiments defined; this session runs {n}: {SESSION_EXPERIMENTS}")
print(f"rough GPU time: ~{n} x 12-20 min, two at a time on T4 x2 -> about {n * 16 / 2 / 60:.1f} h")

## N4. Pipeline code (written — no changes needed)

In [ ]:
%%writefile ner_common.py
"""Phase 2 (L-NER) pipeline: data, label-balanced validation split, tokenisation with EXACT character offsets
(punctuation split before BPE), overlapping windows, BIO tags, span decoding, strict metrics (same rules as
evaluate_phase2.py), Phase-1 backbones that return hidden states, prediction and submission files."""
import glob, json, math, os, re, sys, zipfile
from collections import Counter
import numpy as np, torch, torch.nn as nn

INPUT_ROOT = os.environ.get("P2_INPUT", "/kaggle/input")      # read-only inputs (data prep, Phase-1 checkpoints)
WORK = os.environ.get("P2_WORK", "/kaggle/working")           # everything this notebook writes
DATA = f"{WORK}/p2data"                                        # Phase-2 files downloaded by cell N3

def find_input(pattern):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"'{pattern}' not found under {INPUT_ROOT} — is the right notebook/dataset added as input?")
    return hits[0]

sys.path.insert(0, os.path.dirname(find_input("bpe_tok.py")))
import bpe_tok

# ------------------------------------------------------------------------------------------------ labels / tags
LABELS = [x.strip() for x in open(f"{DATA}/labels.txt", encoding="utf-8") if x.strip()]
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]     # BIO: 1 + 2 x 12 = 25 tags
TAG2ID = {t: i for i, t in enumerate(TAGS)}
IGNORE = -100                                                         # label for tokens that cover no character

# ------------------------------------------------------------------------------------------------ data
def load_json(name):
    return json.load(open(f"{DATA}/{name}", encoding="utf-8"))

def load_data():
    """texts[id] = str, gold[id] = [[start, end, label, text], ...] for the 70 training docs; test[id] = str."""
    texts, gold, fold = {}, {}, {}
    for f in ("fold1", "fold2"):
        t, g = load_json(f"{f}-text.json"), load_json(f"{f}-gold.json")
        texts.update({str(k): v for k, v in t.items()})
        gold.update({str(k): [list(e) for e in v] for k, v in g.items()})
        fold.update({str(k): f for k in t})
    test = {str(k): v for k, v in load_json("fold3-text.json").items()}
    return texts, gold, fold, test

def make_split(gold, n_val=14, target=0.2):
    """Greedy, deterministic: pick n_val documents so that every label has close to `target` of its entities
    in validation (documents are tried in sorted-id order, so the same split comes out every time)."""
    per_doc = {d: Counter(e[2] for e in ents) for d, ents in gold.items()}
    total = sum(per_doc.values(), Counter())
    chosen, cur = [], Counter()
    while len(chosen) < n_val:
        best = None
        for d in sorted(per_doc):
            if d in chosen:
                continue
            c = cur + per_doc[d]
            score = sum((c[l] / total[l] - target) ** 2 for l in LABELS if total[l])
            if best is None or score < best[0]:
                best = (score, d)
        chosen.append(best[1]); cur += per_doc[best[1]]
    val = sorted(chosen)
    train = sorted(d for d in gold if d not in chosen)
    share = {l: round(cur[l] / total[l], 3) if total[l] else None for l in LABELS}
    return train, val, share

# ------------------------------------------------------------------------------------------------ tokenisation
MARK_BYTES = len("▁".encode("utf-8"))
PIECE_RE = re.compile(r"\w+|[^\w\s]")

class SplitTokenizer:
    """Your Phase-1 BPE, applied to smaller pieces: each whitespace word is first split where letters/digits
    meet punctuation ("IPC," -> "IPC" + ","). The first piece keeps the word-start marker, later pieces don't,
    so tokens stay close to pretraining. Every token gets its exact (start, end) character span."""
    def __init__(self, bpe):
        self.bpe, self.stoi, self.unk = bpe, bpe.stoi, bpe.stoi["<unk>"]
        self.cache = {}

    def _pieces(self, s, first):
        key = (s, first)
        out = self.cache.get(key)
        if out is None:
            symbols = list(bpe_tok.word_to_symbols(("▁" if first else "") + s))
            while len(symbols) > 1:
                ranked = [(self.bpe.merge_rank[p], i) for i, p in enumerate(zip(symbols, symbols[1:]))
                          if p in self.bpe.merge_rank]
                if not ranked:
                    break
                _, i = min(ranked)
                symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]
            out = [(self.stoi.get(t, self.unk), len(t)) for t in symbols]
            if len(self.cache) < 2_000_000:
                self.cache[key] = out
        return out

    def encode(self, text):
        ids, spans, _, _ = self.encode_full(text)
        return ids, spans

    def encode_full(self, text):
        """ids, token spans, piece index of each token, and the (start, end) span of each piece
        (a piece = a run of letters/digits or one punctuation character inside a whitespace word)."""
        ids, spans, piece_of, piece_spans = [], [], [], []
        for m in re.finditer(r"\S+", text):
            w, ws = m.group(), m.start()
            for j, p in enumerate(PIECE_RE.finditer(w)):
                part, ps, first = p.group(), ws + p.start(), j == 0
                piece_spans.append((ps, ps + len(part)))
                raw, b = part.encode("utf-8"), 0
                for tid, nbytes in self._pieces(part, first):
                    k0 = max(0, b - (MARK_BYTES if first else 0)); b += nbytes
                    k1 = max(0, b - (MARK_BYTES if first else 0))
                    ids.append(tid)
                    spans.append((ps + len(raw[:k0].decode("utf-8", "ignore")),
                                   ps + len(raw[:k1].decode("utf-8", "ignore"))))
                    piece_of.append(len(piece_spans) - 1)
        return ids, spans, piece_of, piece_spans

def lead_tokens(spans, piece_of, n_pieces):
    """index of the first token of each piece that covers at least one character (-1 if none)"""
    lead = [-1] * n_pieces
    for i, ((s, e), p) in enumerate(zip(spans, piece_of)):
        if s < e and lead[p] < 0:
            lead[p] = i
    return lead

def piece_level_tags(tags, lead):
    """align="piece": only the first token of each piece keeps its tag (the rest are ignored in training)"""
    out = [IGNORE] * len(tags)
    for i in lead:
        if i >= 0:
            out[i] = tags[i]
    return out

def doc_tags(spans, ents):
    """BIO tag id per token. Tokens covering no character (a lone word-start marker) get IGNORE."""
    tags = [TAG2ID["O"] if s < e else IGNORE for s, e in spans]
    starts = np.array([s for s, _ in spans]); ends = np.array([e for _, e in spans])
    for s, e, l, *_ in ents:
        idx = np.nonzero((starts < e) & (ends > s) & (starts < ends))[0]
        for k, i in enumerate(idx):
            tags[i] = TAG2ID[("B-" if k == 0 else "I-") + l]
    return tags

def reachable(spans, ents):
    """How many gold entities can be reproduced exactly from token boundaries (the ceiling for strict F1)."""
    st, en = {s for s, e in spans if s < e}, {e for s, e in spans if s < e}
    return sum(1 for s, e, *_ in ents if s in st and e in en)

# ------------------------------------------------------------------------------------------------ windows
def windows(n, win, stride):
    """(start, end) token ranges covering 0..n; the last window is aligned to the end of the document."""
    if n <= win:
        return [(0, n)]
    out = list(range(0, n - win, stride)) + [n - win]
    return [(s, s + win) for s in sorted(set(out))]

def train_windows(docs, win, stride):
    """docs: {id: (ids, tags)} -> list of (ids_list, tags_list) windows."""
    out = []
    for d, (ids, tags) in docs.items():
        if not ids:
            continue
        for s, e in windows(len(ids), win, stride):
            out.append((ids[s:e], tags[s:e]))
    return out

def batches(wins, batch_size, shuffle, rng):
    """Batches of windows with EQUAL length (no padding: your models have no padding mask)."""
    by_len = {}
    for w in wins:
        by_len.setdefault(len(w[0]), []).append(w)
    out = []
    for L_, group in by_len.items():
        idx = rng.permutation(len(group)) if shuffle else np.arange(len(group))
        for k in range(0, len(group), batch_size):
            out.append([group[i] for i in idx[k:k + batch_size]])
    if shuffle:
        out = [out[i] for i in rng.permutation(len(out))]
    return out

@torch.no_grad()
def predict_logits(model, ids, win, stride, device, batch_size=8):
    """Tag scores for every token of one document. Each token's scores come from the window in which it has
    the most context on both sides (its distance to the nearest window edge is largest)."""
    n = len(ids)
    out = np.zeros((n, len(TAGS)), dtype=np.float32)
    if n == 0:                                         # empty document: nothing to predict
        return out
    W = windows(n, win, stride)
    best = np.full(n, -1)
    x_all = torch.tensor(ids, dtype=torch.long)
    for k in range(0, len(W), batch_size):
        chunk = W[k:k + batch_size]
        x = torch.stack([x_all[s:e] for s, e in chunk]).to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x).float().cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            pos = np.arange(s, e)
            left = pos - s if s > 0 else np.full(e - s, n)
            right = e - 1 - pos if e < n else np.full(e - s, n)
            score = np.minimum(left, right)
            take = score > best[s:e]
            out[s:e][take] = logits[j][take]
            best[s:e][take] = score[take]
    return out

# ------------------------------------------------------------------------------------------------ decoding
def decode(tag_ids, spans, mode="lenient"):
    """Token tags -> [[start, end, label], ...] character spans.
    lenient: an I-X that does not continue an X entity starts a new X entity.
    strict:  such an orphan I-X is ignored (only B-X can start an entity)."""
    ents, cur = [], None
    for t, (s, e) in zip(tag_ids, spans):
        if s >= e:
            continue                                   # token covers no character
        name = TAGS[t]
        if name == "O":
            if cur: ents.append(cur); cur = None
            continue
        bio, lab = name.split("-", 1)
        if bio == "I" and cur is not None and cur[2] == lab:
            cur[1] = e
            continue
        if cur: ents.append(cur); cur = None
        if bio == "B" or mode == "lenient":
            cur = [s, e, lab]
    if cur:
        ents.append(cur)
    seen, out = set(), []
    for s, e, l in ents:                               # the evaluator rejects duplicate spans
        if (s, e, l) not in seen:
            seen.add((s, e, l)); out.append([int(s), int(e), l])
    return out

# ------------------------------------------------------------------------------------------------ metrics
def strict_scores(gold, pred):
    """Same rules as evaluate_phase2.py: exact (start, end, label) match, macro over the 12 labels."""
    stats = {l: [0, 0, 0] for l in LABELS}
    for d in gold:
        g = {(s, e, l) for s, e, l, *_ in gold[d]}
        p = {(s, e, l) for s, e, l, *_ in pred.get(d, [])}
        for l in LABELS:
            gs, ps = {x for x in g if x[2] == l}, {x for x in p if x[2] == l}
            stats[l][0] += len(gs & ps); stats[l][1] += len(ps - gs); stats[l][2] += len(gs - ps)
    per = {}
    for l, (tp, fp, fn) in stats.items():
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[l] = {"P": pr, "R": rc, "F1": 2 * pr * rc / (pr + rc) if pr + rc else 0.0, "tp": tp, "fp": fp, "fn": fn}
    mean = lambda k: float(np.mean([per[l][k] for l in LABELS]))
    return {"macro_f1": mean("F1"), "macro_p": mean("P"), "macro_r": mean("R"), "per_label": per}

def error_breakdown(gold, pred):
    """Classify every mistake: right span + wrong label, right label + wrong boundary, spurious, missed."""
    c = Counter()
    for d in gold:
        g = {(s, e, l) for s, e, l, *_ in gold[d]}; p = {(s, e, l) for s, e, l, *_ in pred.get(d, [])}
        g_sp = {(s, e): l for s, e, l in g}
        for s, e, l in p - g:
            if (s, e) in g_sp:
                c["wrong label (span exact)"] += 1
            elif any(l2 == l and s < e2 and s2 < e for s2, e2, l2 in g):
                c["wrong boundary (label right)"] += 1
            else:
                c["spurious entity"] += 1
        p_sp = {(s, e) for s, e, _ in p}
        for s, e, l in g - p:
            if (s, e) not in p_sp and not any(l2 == l and s < e2 and s2 < e for s2, e2, l2 in p):
                c["missed entity"] += 1
    return dict(c)

# ------------------------------------------------------------------------------------------------ backbones
TINY = {   # used only by the CPU smoke test (random weights, no checkpoint)
    "decoder":   dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "prefix":    dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "encdec":    dict(d_model=32, n_enc_layers=2, n_dec_layers=1, n_heads=4, max_len=256, max_tgt_len=32, dropout=0.0),
    "swa":       dict(d_model=32, n_layers=2, n_heads=4, window=16, max_len=256, dropout=0.0),
    "diffusion": dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "moe":       dict(d_model=32, n_layers=2, n_heads=4, d_ff=64, n_experts=4, max_len=256, dropout=0.0),
}

class Backbone(nn.Module):
    """Wraps one of YOUR Phase-1 models so that backbone(ids) returns hidden states (B, T, d_model) instead of
    next-token scores. Your model code is not changed: its output layer is replaced by Identity, so forward()
    returns the final-LayerNorm output. For encdec the ENCODER output is used (read with a hook).
    context="echo" (causal models): the window is fed twice and the second copy's states are used, so every
    token has seen the whole window (Springer et al. 2024, "Repetition Improves Language Model Embeddings")."""
    def __init__(self, arch, lm, bos_id, context="plain"):
        super().__init__()
        self.arch, self.lm, self.bos_id, self.context = arch, lm, bos_id, context
        self.kind = getattr(lm, "kind", "causal")
        self.d_model = lm.tok_emb.embedding_dim
        self.max_input = lm.max_len // 2 if context == "echo" else lm.max_len
        lm.head = nn.Identity()
        if arch == "encdec":
            # the NER head reads the ENCODER only: drop the decoder parts (unused, would waste memory and be
            # counted as trainable parameters)
            del lm.dec_layers, lm.dec_norm, lm.tgt_pos_emb

    def forward(self, ids):
        T0 = ids.size(1)
        x = torch.cat([ids, ids], 1) if self.context == "echo" else ids
        T = x.size(1)
        if self.arch == "encdec":                 # your EncDecLM's encoder path: embeddings -> enc_layers -> enc_norm
            lm = self.lm
            h = lm.tok_emb(x) + lm.src_pos_emb(torch.arange(T, device=x.device))
            for layer in lm.enc_layers:
                h = layer(h)
            h = lm.enc_norm(h)
        elif self.kind == "prefix":
            h = self.lm(x, prefix_len=T)          # whole input = prefix -> fully bidirectional
        else:
            h = self.lm(x)
        return h[:, T - T0:] if self.context == "echo" else h

def build_lm(arch, cfg):
    import models, models_x
    return (models_x if arch in ("diffusion", "moe") else models).build_model(arch, vocab_size=len(bpe_tok_bpe().itos), **cfg)

_BPE = None
def bpe_tok_bpe():
    global _BPE
    if _BPE is None:
        _BPE = bpe_tok.load_bpe(find_input("bpe.json"))
    return _BPE

def load_backbone(arch, ckpt_pattern, context="plain", smoke=False):
    """Rebuilds your Phase-1 model from its checkpoint (its saved cfg + weights) and wraps it."""
    bpe = bpe_tok_bpe()
    if smoke:
        lm = build_lm(arch, TINY[arch])
    else:
        ck = torch.load(find_input(ckpt_pattern), map_location="cpu", weights_only=False)
        assert ck.get("arch", arch) == arch, f"checkpoint {ckpt_pattern} is a '{ck.get('arch')}' model, not '{arch}'"
        lm = build_lm(arch, ck["cfg"])
        lm.load_state_dict(ck["model"])
    return Backbone(arch, lm, bpe.stoi["<s>"], context)

def iter_adaptable(module, skip=("head",)):
    """Every weight a low-rank method (LoRA / LoRMA) can adapt inside `module`:
    yields (name, owner_module, parameter_name, weight) for nn.Linear weights and the packed q/k/v weight of
    nn.MultiheadAttention ('in_proj_weight'). Names containing an entry of `skip` are left out."""
    for name, mod in module.named_modules():
        if any(s in name.split(".") for s in skip):
            continue
        if isinstance(mod, nn.MultiheadAttention) and getattr(mod, "in_proj_weight", None) is not None:
            yield name, mod, "in_proj_weight", mod.in_proj_weight
        elif isinstance(mod, nn.Linear):
            yield name, mod, "weight", mod.weight

def layer_stack(backbone):
    """(attribute name, list of transformer layers, final LayerNorm) of the part of the model the NER head reads:
    decoder/prefix/swa/diffusion -> lm.layers + lm.norm ; moe -> lm.blocks + lm.norm ; encdec -> lm.enc_layers + lm.enc_norm"""
    lm = backbone.lm
    if backbone.arch == "encdec":
        return "enc_layers", lm.enc_layers, lm.enc_norm
    if hasattr(lm, "blocks"):
        return "blocks", lm.blocks, lm.norm
    return "layers", lm.layers, lm.norm

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train

# ------------------------------------------------------------------------------------------------ submission
def write_predictions(preds, ids, path_json, path_zip=None):
    out = {d: preds.get(d, []) for d in ids}
    json.dump(out, open(path_json, "w", encoding="utf-8"), ensure_ascii=False)
    if path_zip:
        with zipfile.ZipFile(path_zip, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(path_json, arcname="prediction.json")

## N5. Your Phase-1 model code (unchanged copies of your `models.py` and `models_x.py`)

In [ ]:
%%writefile models.py
# =====================================================================================================
#  CS-781 Phase 1 — four language models                      
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
#  PART 1 — MASKS
# =====================================================================================================

def causal_mask(T, device):
    mask = torch.ones(T, T, dtype=torch.bool, device=device).triu(diagonal=1)
    return mask


def prefix_mask(T, P, device):
    mask = causal_mask(T, device)
    mask[:, :P] = False
    return mask


def alibi_window_bias(T, n_heads, window, device):
    i = torch.arange(T, device=device)[:, None]
    j = torch.arange(T, device=device)[None, :]

    dist = i - j

    slopes = torch.tensor(
        [2 ** (-8 * (h + 1) / n_heads) for h in range(n_heads)],
        device=device,
        dtype=torch.float32,
    )[:, None, None]

    bias = -slopes * dist

    bias = bias.masked_fill(
        (dist < 0) | (dist >= window),
        float("-inf")
    )

    return bias


# =====================================================================================================
#  PART 2 — SHARED HELPERS
# =====================================================================================================

def make_layers(n_layers, d_model, n_heads, dropout):

    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
#  MODEL 1 — DECODER-ONLY (GPT-style)
# =====================================================================================================

class DecoderLM(nn.Module):
    kind = "causal"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = causal_mask(T, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 2 — PREFIX-LM
# =====================================================================================================

class PrefixLM(nn.Module):
    kind = "prefix"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, prefix_len):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = prefix_mask(T, prefix_len, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 3 — ENCODER-DECODER
# =====================================================================================================

class EncDecLM(nn.Module):
    kind = "encdec"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_enc_layers,
        n_dec_layers,
        n_heads,
        max_len,
        max_tgt_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.max_tgt_len = max_len, max_tgt_len

        # Shared token embedding
        self.tok_emb = nn.Embedding(vocab_size, d_model)

        # Separate position embeddings
        self.src_pos_emb = nn.Embedding(max_len, d_model)
        self.tgt_pos_emb = nn.Embedding(max_tgt_len, d_model)

        # Encoder
        self.enc_layers = make_layers(
            n_enc_layers,
            d_model,
            n_heads,
            dropout
        )

        self.enc_norm = nn.LayerNorm(d_model)

        # Decoder
        self.dec_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(n_dec_layers)
        ])

        self.dec_norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, src, tgt_in):
        S, U = src.size(1), tgt_in.size(1)

        # -------------------------
        # ENCODER
        # -------------------------

        src_pos = torch.arange(S, device=src.device)

        mem = self.tok_emb(src) + self.src_pos_emb(src_pos)

        for layer in self.enc_layers:
            mem = layer(mem)

        mem = self.enc_norm(mem)

        # -------------------------
        # DECODER
        # -------------------------

        tgt_pos = torch.arange(U, device=tgt_in.device)

        y = self.tok_emb(tgt_in) + self.tgt_pos_emb(tgt_pos)

        tgt_mask = causal_mask(U, tgt_in.device)

        for layer in self.dec_layers:
            y = layer(
                y,
                mem,
                tgt_mask=tgt_mask
            )

        return self.head(self.dec_norm(y))


# =====================================================================================================
#  MODEL 4 — EFFICIENT: SLIDING-WINDOW ATTENTION + ALiBi
# =====================================================================================================

class SlidingWindowLM(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        window,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.n_heads, self.window = max_len, n_heads, window

        self.tok_emb = nn.Embedding(vocab_size, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        # Token embeddings only
        x = self.tok_emb(idx)

        # ALiBi + sliding-window bias
        bias = alibi_window_bias(
            T,
            self.n_heads,
            self.window,
            idx.device
        )

        # (n_heads, T, T)
        # -> (B * n_heads, T, T)
        bias = bias.repeat(B, 1, 1)

        for layer in self.layers:
            x = layer(
                x,
                src_mask=bias
            )

        return self.head(self.norm(x))


# =====================================================================================================
#  FACTORY — do not change.
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "decoder": DecoderLM,
        "prefix": PrefixLM,
        "encdec": EncDecLM,
        "swa": SlidingWindowLM
    }

    return classes[arch](vocab_size, **cfg)

In [ ]:
%%writefile models_x.py

# =====================================================================================================
# CS-781 Phase 1 — two extra architectures
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
# PART 0 — HELPERS
# =====================================================================================================

def causal_mask(T, device):
    return torch.ones(
        T, T,
        dtype=torch.bool,
        device=device
    ).triu(diagonal=1)


def make_layers(n_layers, d_model, n_heads, dropout):
    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(
            module.weight,
            mean=0.0,
            std=0.02
        )

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
# MODEL 5 — LexDiffusionLM
# =====================================================================================================

class LexDiffusionLM(nn.Module):
    kind = "diffusion"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # Transformer layers
        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        # Final normalization
        self.norm = nn.LayerNorm(d_model)

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Fully bidirectional attention.
        # NO MASK.
        for layer in self.layers:
            x = layer(x)

        # Final normalization + output
        x = self.norm(x)

        return self.head(x)


# =====================================================================================================
# MODEL 6 — LexMoE
# =====================================================================================================

class MoEFeedForward(nn.Module):
    """
    Top-1 Mixture-of-Experts feed-forward network.

    Each token is routed to exactly one expert.
    """

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        self.n_experts = n_experts

        # Filled during forward()
        self.aux_loss = None
        self.last_choice = None

        # Router
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False
        )

        # Experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Linear(d_ff, d_model),
                nn.Dropout(dropout),
            )
            for _ in range(n_experts)
        ])


    def forward(self, x):
        B, T, D = x.shape

        # Flatten tokens
        flat = x.reshape(
            B * T,
            D
        )

        # Router probabilities
        router_logits = self.router(flat)

        # Float32 router softmax for numerical stability
        probs = torch.softmax(
            router_logits.float(),
            dim=-1
        )

        # Best expert for each token
        top_p, top_i = probs.max(
            dim=-1
        )

        # Output buffer
        out = flat.new_zeros(
            flat.shape
        )

        # ------------------------------------------------
        # Route tokens to experts
        # ------------------------------------------------

        for e in range(self.n_experts):

            sel = (top_i == e)

            if sel.any():

                expert_out = self.experts[e](
                    flat[sel]
                )

                expert_out = (
                    expert_out
                    * top_p[sel][:, None]
                )

                out[sel] = expert_out.to(
                    out.dtype
                )


        # ------------------------------------------------
        # Switch load-balancing loss
        # ------------------------------------------------

        counts = torch.bincount(
            top_i,
            minlength=self.n_experts
        ).float()

        frac_e = counts / flat.shape[0]

        prob_e = probs.mean(
            dim=0
        )

        self.aux_loss = (
            self.n_experts
            * torch.sum(frac_e * prob_e)
        )


        # Store routing decisions
        self.last_choice = top_i.reshape(
            B,
            T
        )


        # Restore original shape
        return out.reshape(
            B,
            T,
            D
        )


# =====================================================================================================
# MoE BLOCK
# =====================================================================================================

class MoEBlock(nn.Module):
    """
    Pre-norm Transformer block:

        x -> LayerNorm -> Attention -> residual
          -> LayerNorm -> MoE      -> residual
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        # Pre-norm attention
        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.drop = nn.Dropout(
            dropout
        )

        # Pre-norm MoE
        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.moe = MoEFeedForward(
            d_model,
            d_ff,
            n_experts,
            dropout
        )


    def forward(self, x, mask):

        # ------------------------------------------------
        # Attention
        # ------------------------------------------------

        h = self.norm1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=mask,
            need_weights=False
        )

        x = x + self.drop(
            attn_out
        )


        # ------------------------------------------------
        # MoE feed-forward
        # ------------------------------------------------

        moe_out = self.moe(
            self.norm2(x)
        )

        x = x + moe_out

        return x


# =====================================================================================================
# LexMoE
# =====================================================================================================

class LexMoE(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        n_experts,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Filled during forward()
        self.aux_loss = None

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # MoE Transformer blocks
        self.blocks = nn.ModuleList([
            MoEBlock(
                d_model,
                n_heads,
                d_ff,
                n_experts,
                dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(
            d_model
        )

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Causal mask
        mask = causal_mask(
            T,
            idx.device
        )

        # Store auxiliary losses
        aux_losses = []

        # Transformer blocks
        for block in self.blocks:

            x = block(
                x,
                mask
            )

            aux_losses.append(
                block.moe.aux_loss
            )

        # Average auxiliary loss across layers
        self.aux_loss = torch.stack(
            aux_losses
        ).mean()

        # Final normalization
        x = self.norm(x)

        # Vocabulary logits
        return self.head(x)


# =====================================================================================================
# FACTORY — DO NOT CHANGE
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "diffusion": LexDiffusionLM,
        "moe": LexMoE
    }

    return classes[arch](
        vocab_size,
        **cfg
    )

## N6. `ner_methods.py` — YOUR CODE
Read the header of the cell first: it explains what the pipeline gives you, what it does around your code, and the
tools you need. Then fill the gaps in the order listed, testing each step with N8 (CPU, no GPU needed).

In [ ]:
%%writefile ner_methods.py
# =====================================================================================================
#  CS-781 Phase 2 — YOUR CODE: the NER head, the fine-tuning methods, and knowledge distillation
#                                                                AUTHOR: <your name>   (your own code)
# =====================================================================================================

import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import parametrize
import ner_common as L


# =====================================================================================================
#  1. THE NER HEAD
# =====================================================================================================

class TokenClassifier(nn.Module):

    def __init__(self, backbone, n_tags, dropout):
        super().__init__()

        # Keep backbone
        self.backbone = backbone

        # Dropout before classifier
        self.dropout = nn.Dropout(dropout)

        # Token-level classification head
        self.head = nn.Linear(
            backbone.d_model,
            n_tags
        )


    def forward(self, ids):

        # Backbone hidden states
        h = self.backbone(ids)

        # Classification
        return self.head(
            self.dropout(h)
        )


# =====================================================================================================
#  2. SIMPLE METHODS: full, partial, BitFit
# =====================================================================================================

def method_full(model, **cfg):
    """Full fine-tuning: every parameter of the backbone trains."""

    for p in model.backbone.parameters():
        p.requires_grad_(True)


def method_partial(model, n_last=2, **cfg):
    """
    Partial / layer-wise fine-tuning:
    only the LAST n_last transformer layers + final LayerNorm train.
    """

    # Get the relevant layer stack
    name, layers, final_norm = L.layer_stack(
        model.backbone
    )

    # Train only the last n_last layers
    for layer in layers[-n_last:]:
        for p in layer.parameters():
            p.requires_grad_(True)

    # Train final LayerNorm
    for p in final_norm.parameters():
        p.requires_grad_(True)


def method_bitfit(model, **cfg):
    """
    BitFit:
    only bias parameters of the backbone train.
    """

    for name, p in model.backbone.named_parameters():

        if name.endswith("bias"):
            p.requires_grad_(True)


# =====================================================================================================
#  3. LoRA
# =====================================================================================================

class LoRAParam(nn.Module):
    """
    Additive low-rank update:

        W' = W + (alpha / r) * B @ A
    """

    def __init__(
        self,
        d_out,
        d_in,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.empty(r, d_in)
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=math.sqrt(5)
        )

        # B: zero initialization
        # This guarantees W' = W initially.
        self.B = nn.Parameter(
            torch.zeros(d_out, r)
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        return W + self.scale * (
            self.B @ self.A
        )


def method_lora(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRAParam(
                d_out,
                d_in,
                r,
                alpha
            )
        )


# =====================================================================================================
#  4. LoRMA
# =====================================================================================================

class LoRMAParam(nn.Module):
    """
    LoRMA+ multiplicative low-rank update:

        W' = W + (alpha / r) * B @ (A @ W)

    where:

        A : (r, d_out)
        B : (d_out, r)
    """

    def __init__(
        self,
        d_out,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.randn(
                r,
                d_out
            ) * 0.02
        )

        # B: zero initialization
        self.B = nn.Parameter(
            torch.zeros(
                d_out,
                r
            )
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        # First compute A @ W
        AW = self.A @ W

        # Then B @ (A @ W)
        update = self.B @ AW

        return W + self.scale * update


def method_lorma(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRMAParam(
                d_out,
                r,
                alpha
            )
        )


# =====================================================================================================
#  5. KNOWLEDGE DISTILLATION
# =====================================================================================================

def kd_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=2.0,
    alpha=0.5
):
    """
    Combined hard-label cross entropy and
    teacher-student KL divergence.
    """

    # Only real labelled tokens
    mask = labels != -100

    s = student_logits[mask]
    t = teacher_logits[mask]
    y = labels[mask]

    # --------------------------------------------------------
    # Hard-label loss
    # --------------------------------------------------------

    hard = F.cross_entropy(
        s,
        y
    )

    # --------------------------------------------------------
    # Soft teacher/student distributions
    # --------------------------------------------------------

    soft = F.kl_div(
        F.log_softmax(
            s / temperature,
            dim=-1
        ),
        F.softmax(
            t / temperature,
            dim=-1
        ),
        reduction="batchmean"
    )

    # Hinton temperature scaling
    soft = soft * (
        temperature * temperature
    )

    # Combined loss
    return (
        alpha * hard
        + (1.0 - alpha) * soft
    )


def make_student(
    backbone,
    keep_layers
):
    """
    Make a smaller copy of a Phase-1 backbone,
    keeping only its first keep_layers layers.
    """

    # Deep copy so original backbone is untouched
    small = copy.deepcopy(
        backbone
    )

    # Find the layer stack
    name, layers, _ = L.layer_stack(
        small
    )

    # Keep only first keep_layers
    setattr(
        small.lm,
        name,
        nn.ModuleList(
            list(layers)[:keep_layers]
        )
    )

    return small


# =====================================================================================================
#  METHOD TABLE
# =====================================================================================================

METHODS = {
    "full": method_full,
    "partial": method_partial,
    "bitfit": method_bitfit,
    "lora": method_lora,
    "lorma": method_lorma,
}

## N7. Training script (written — runs one experiment)

In [ ]:
%%writefile train_ner.py
"""Runs ONE experiment from run_config_ner.json: builds your Phase-1 backbone + your NER head, applies your
fine-tuning method, trains on the training documents, keeps the epoch with the best validation strict
macro-F1, then writes validation + test predictions and the Codabench zip.
   python train_ner.py --exp NAME --device cuda:0        (real run)
   python train_ner.py --exp NAME --device cpu --smoke   (tiny random model, few documents: checks the code)"""
import argparse, json, math, os, random, time
import numpy as np, torch, torch.nn.functional as F
import ner_common as L
import ner_methods as M

ap = argparse.ArgumentParser()
ap.add_argument("--exp", required=True); ap.add_argument("--device", default="cuda:0")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
dev = args.device
if dev.startswith("cuda"):
    torch.cuda.set_device(dev)

C = json.load(open(f"{L.WORK}/run_config_ner.json"))
def exp_cfg(name):
    if name in C["experiments"]:
        return {**C["defaults"], **C["experiments"][name], "name": name}
    raise KeyError(f"experiment '{name}' is not in EXPERIMENTS (cell N3)")
E = exp_cfg(args.exp)
if args.smoke:
    E.update(window=96, stride=64, epochs=1, batch_size=4, patience=99)
RUNS = f"{L.WORK}/{'ner_runs_smoke' if args.smoke else 'ner_runs'}"
OUT = f"{RUNS}/{args.exp}"; os.makedirs(OUT, exist_ok=True)
log = open(f"{OUT}/log.jsonl", "w")
t0 = time.time()
def say(**kw):
    kw["t"] = round(time.time() - t0); log.write(json.dumps(kw) + "\n"); log.flush()
    print(args.exp, json.dumps(kw), flush=True)
seed = C["seed"]; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------------------------------------------ data
texts, gold, fold, test = L.load_data()
train_ids, val_ids, share = L.make_split(gold, n_val=C["n_val_docs"], target=C["val_fraction"])
if E["train_on_all"]:
    train_ids = sorted(gold)
if args.smoke:
    train_ids, val_ids, test = train_ids[:4], val_ids[:2], dict(list(test.items())[:2])
tok = L.SplitTokenizer(L.bpe_tok_bpe())
enc = {d: tok.encode_full(t) for d, t in {**texts, **test}.items() if d in set(train_ids) | set(val_ids) | set(test)}
lead = {d: L.lead_tokens(enc[d][1], enc[d][2], len(enc[d][3])) for d in enc}
def train_tags(d):
    t = L.doc_tags(enc[d][1], gold[d])
    return L.piece_level_tags(t, lead[d]) if E["align"] == "piece" else t
train_docs = {d: (enc[d][0], train_tags(d)) for d in train_ids}

# ------------------------------------------------------------------ model
def build(e, smoke):
    backbone = L.load_backbone(e["arch"], C["checkpoints"][e["arch"]], context=e["context"], smoke=smoke)
    if e.get("student_layers"):
        backbone = M.make_student(backbone, e["student_layers"])
    model = M.TokenClassifier(backbone, n_tags=len(L.TAGS), dropout=e["head_dropout"])
    for p in model.parameters():
        p.requires_grad_(False)
    M.METHODS[e["method"]](model, **e["method_cfg"])
    for p in model.head.parameters():          # the new head is always trained
        p.requires_grad_(True)
    return model

model = build(E, args.smoke).to(dev)
def fit_window(max_input):
    """window capped at what the model accepts; the stride keeps the same overlap ratio as configured"""
    w = min(E["window"], max_input)
    return w, max(1, min(E["stride"], round(w * E["stride"] / E["window"])))
win, stride = fit_window(model.backbone.max_input)
total, trainable = L.count_params(model)
say(event="start", arch=E["arch"], method=E["method"], params_total=total, params_trainable=trainable,
    trainable_pct=round(100 * trainable / total, 3), window=win, stride=stride,
    train_docs=len(train_ids), val_docs=len(val_ids), val_label_share=share)

teacher = None
if E.get("teacher"):
    TE = exp_cfg(E["teacher"])
    tpath = f"{RUNS}/{E['teacher']}/best.pt"
    if not os.path.exists(tpath):
        tpath = L.find_input(f"ner_runs/{E['teacher']}/best.pt")      # teacher trained in an earlier session
    teacher = build(TE, args.smoke)
    teacher.load_state_dict(torch.load(tpath, map_location="cpu", weights_only=False)["state"])
    teacher.to(dev).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    win, stride = fit_window(min(win, teacher.backbone.max_input))
    say(event="teacher", name=E["teacher"], params_total=L.count_params(teacher)[0])

# ------------------------------------------------------------------ training setup
wins = L.train_windows(train_docs, win, stride)
tag_counts = np.bincount([t for _, tags in wins for t in tags if t != L.IGNORE], minlength=len(L.TAGS))
weights = None
if E["class_weights"] == "inv_sqrt":
    w = 1.0 / np.sqrt(np.maximum(tag_counts, 1)); w = w / w.mean()
    weights = torch.tensor(w, dtype=torch.float32, device=dev)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=E["lr"], weight_decay=E["weight_decay"])
rng = np.random.RandomState(seed)
steps_per_epoch = len(L.batches(wins, E["batch_size"], False, rng))
if args.smoke:
    steps_per_epoch = min(steps_per_epoch, 6)
total_steps = steps_per_epoch * E["epochs"]
warm = max(1, int(E["warmup_frac"] * total_steps))
def lr_at(step):                                   # linear warm-up, then linear decay to 0
    return E["lr"] * min((step + 1) / warm, max(0.0, (total_steps - step) / max(1, total_steps - warm)))
scaler = torch.amp.GradScaler("cuda", enabled=dev.startswith("cuda"))

def decode_doc(d, logits):
    if E["align"] == "piece":                  # one tag per piece, taken from its first token
        keep = [(i, sp) for i, sp in zip(lead[d], enc[d][3]) if i >= 0]
        return L.decode([int(logits[i].argmax()) for i, _ in keep], [sp for _, sp in keep], E["decode"])
    return L.decode(logits.argmax(-1), enc[d][1], E["decode"])

def evaluate(doc_ids, keep_logits=None):
    model.eval()
    preds = {}
    for d in doc_ids:
        logits = L.predict_logits(model, enc[d][0], win, stride, dev, E["batch_size"])
        preds[d] = decode_doc(d, logits)
        if keep_logits is not None:
            keep_logits[d] = logits.astype(np.float16)
    model.train()
    return preds

# ------------------------------------------------------------------ train
best, best_epoch, bad, step = -1.0, -1, 0, 0
model.train()
if dev.startswith("cuda"):
    torch.cuda.reset_peak_memory_stats()
for epoch in range(1, E["epochs"] + 1):
    te, losses = time.time(), []
    for bi, batch in enumerate(L.batches(wins, E["batch_size"], True, rng)):
        if args.smoke and bi >= steps_per_epoch:
            break
        x = torch.tensor([w[0] for w in batch], dtype=torch.long, device=dev)
        y = torch.tensor([w[1] for w in batch], dtype=torch.long, device=dev)
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x).float()
            if teacher is not None:
                with torch.no_grad():
                    t_logits = teacher(x).float()
                loss = M.kd_loss(logits, t_logits, y, **E["kd"])
            else:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1),
                                       weight=weights, ignore_index=L.IGNORE)
            aux = getattr(model.backbone.lm, "aux_loss", None)
            if E["moe_aux_coef"] and aux is not None:
                loss = loss + E["moe_aux_coef"] * aux
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, E["grad_clip"])
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        losses.append(loss.item()); step += 1
    epoch_sec = time.time() - te
    if E["train_on_all"]:
        say(event="epoch", epoch=epoch, train_loss=round(float(np.mean(losses)), 4), sec=round(epoch_sec))
        torch.save({"state": model.state_dict(), "exp": E, "epoch": epoch}, f"{OUT}/best.pt"); best_epoch = epoch
        continue
    s = L.strict_scores({d: gold[d] for d in val_ids}, evaluate(val_ids))
    say(event="epoch", epoch=epoch, train_loss=round(float(np.mean(losses)), 4), val_macro_f1=round(s["macro_f1"], 4),
        val_macro_p=round(s["macro_p"], 4), val_macro_r=round(s["macro_r"], 4), sec=round(epoch_sec))
    if s["macro_f1"] > best:
        best, best_epoch, bad = s["macro_f1"], epoch, 0
        torch.save({"state": model.state_dict(), "exp": E, "epoch": epoch}, f"{OUT}/best.pt")
    else:
        bad += 1
        if bad >= E["patience"]:
            say(event="early_stop", epoch=epoch)
            break

# ------------------------------------------------------------------ final predictions with the best epoch
model.load_state_dict(torch.load(f"{OUT}/best.pt", map_location=dev, weights_only=False)["state"])
peak = round(torch.cuda.max_memory_allocated() / 1e9, 2) if dev.startswith("cuda") else None
summary = {"exp": args.exp, "arch": E["arch"], "method": E["method"], "align": E["align"], "best_epoch": best_epoch,
           "params_total": total, "params_trainable": trainable, "peak_gpu_gb": peak,
           "train_windows": len(wins), "window": win, "stride": stride}
if not E["train_on_all"]:
    vlog = {}
    vpred = evaluate(val_ids, vlog)
    np.savez_compressed(f"{OUT}/val_logits.npz", **vlog)      # for decoding comparisons on CPU (N13)
    L.write_predictions(vpred, val_ids, f"{OUT}/val_prediction.json")
    json.dump({d: [[s, e, l] for s, e, l, *_ in gold[d]] for d in val_ids}, open(f"{OUT}/val_truth.json", "w"))
    s = L.strict_scores({d: gold[d] for d in val_ids}, vpred)
    summary.update(val_macro_f1=round(s["macro_f1"], 4), val_macro_p=round(s["macro_p"], 4), val_macro_r=round(s["macro_r"], 4),
                   val_per_label_f1={l: round(v["F1"], 4) for l, v in s["per_label"].items()},
                   val_errors=L.error_breakdown({d: gold[d] for d in val_ids}, vpred))
ti = time.time()
tpred = evaluate(list(test))
summary["test_predict_sec"] = round(time.time() - ti, 1)
L.write_predictions(tpred, list(test), f"{OUT}/prediction.json", f"{OUT}/submission_{args.exp}.zip")
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
say(event="done", **{k: v for k, v in summary.items() if k not in ("val_per_label_f1", "val_errors")})

## N8. Checks + CPU smoke test of YOUR code
Part 1 checks the data, the validation split and the checkpoints. Part 2 runs each experiment in `SMOKE_EXPERIMENTS`
on **tiny random models** (no checkpoints, no GPU) for a few steps — it only proves that your code runs end to end.
Every experiment must end with **exit code 0**. Numbers here are meaningless.

In [ ]:
import json, subprocess, sys, importlib, numpy as np
import ner_common as L
importlib.reload(L)
texts, gold, fold, test = L.load_data()
C = json.load(open("/kaggle/working/run_config_ner.json"))
train_ids, val_ids, share = L.make_split(gold, n_val=C["n_val_docs"], target=C["val_fraction"])
print(f"documents: train {len(train_ids)} | validation {len(val_ids)} | test {len(test)}")
print("share of each label's entities in validation:", share)
tok = L.SplitTokenizer(L.bpe_tok_bpe())
n_ent = n_ok = n_ok_piece = 0
for d in gold:
    ids, spans, piece_of, piece_spans = tok.encode_full(texts[d]); n_ent += len(gold[d])
    n_ok += L.reachable(spans, gold[d]); n_ok_piece += L.reachable(piece_spans, gold[d])
print(f"entities reachable exactly from token boundaries: {n_ok}/{n_ent} ({100*n_ok/n_ent:.2f}%)  <- ceiling, align='token'")
print(f"entities reachable exactly from piece boundaries: {n_ok_piece}/{n_ent} ({100*n_ok_piece/n_ent:.2f}%)  <- ceiling, align='piece'")
for arch in sorted({C["experiments"][e]["arch"] for e in C["experiments"]}):
    try:
        path = L.find_input(C["checkpoints"][arch])
        ck = __import__("torch").load(path, map_location="cpu", weights_only=False)
        print(f"checkpoint {arch:9s}: {path} | step {ck.get('step')} | val_loss {ck.get('val_loss', float('nan')):.3f}")
    except Exception as e:
        print(f"checkpoint {arch:9s}: MISSING -> {e}")

In [ ]:
# Part 2: CPU smoke test of your code (tiny random models). Runs the experiments listed in SMOKE_EXPERIMENTS (N3).
import subprocess, sys
for exp in SMOKE_EXPERIMENTS:
    r = subprocess.run([sys.executable, "train_ner.py", "--exp", exp, "--device", "cpu", "--smoke"],
                       capture_output=True, text=True, cwd="/kaggle/working")
    lines = [l for l in (r.stdout + r.stderr).splitlines() if "Warning" not in l and "warn(" not in l]
    print(f"===== {exp}: exit code {r.returncode}")
    print("\n".join(lines[-6:] if r.returncode == 0 else lines[-25:]))

## N9. Launch the session's experiments on the GPUs
One experiment per GPU at a time; when one finishes, the next starts. A distillation experiment waits until its
teacher has finished. Progress (last log line of each running experiment) every 2 minutes.

In [ ]:
import subprocess, sys, time, os, json, torch
C = json.load(open("/kaggle/working/run_config_ner.json"))
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
pending, running, finished = list(SESSION_EXPERIMENTS), {}, {}

def ready(exp):
    t = C["experiments"][exp].get("teacher")
    return not t or (t not in pending and t not in [e for e, _ in running.values()])

last_print = 0
while pending or running:
    for dev in devices:
        if dev in running and running[dev][1].poll() is not None:
            exp, p = running.pop(dev); finished[exp] = p.returncode
            print(time.strftime("%H:%M"), f"| {exp} finished on {dev} with exit code {p.returncode}", flush=True)
        if dev not in running:
            nxt = next((e for e in pending if ready(e)), None)
            if nxt:
                pending.remove(nxt)
                out = open(f"/kaggle/working/{nxt}.out", "w")
                running[dev] = (nxt, subprocess.Popen([sys.executable, "train_ner.py", "--exp", nxt, "--device", dev],
                                                      stdout=out, stderr=subprocess.STDOUT, cwd="/kaggle/working"))
                print(time.strftime("%H:%M"), f"| started {nxt} on {dev}", flush=True)
    if time.time() - last_print > 120:
        for dev, (exp, _) in running.items():
            lines = [l for l in open(f"/kaggle/working/{exp}.out").read().splitlines() if l.startswith(exp)]
            print(time.strftime("%H:%M"), "|", (lines[-1] if lines else f"{exp}: starting...")[:230], flush=True)
        last_print = time.time()
    time.sleep(15)
print("\nexit codes:", finished)
for exp, code_ in finished.items():
    if code_ != 0:
        print(f"\n===== {exp} FAILED — last lines:\n" + open(f"/kaggle/working/{exp}.out").read()[-3000:])

## N10. Results — official evaluator on each experiment's validation predictions + comparison table

In [ ]:
%%writefile evaluate_phase2.py
#!/usr/bin/env python3
"""Local evaluator for Phase 2.

Usage:
  python evaluate_phase2.py --truth truth.json --prediction prediction.json --labels labels.txt
"""
import argparse, json, sys

class DuplicateJSONKeyError(ValueError): pass
def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")
def pairs_no_duplicates(pairs):
    out={}; dup=[]
    for k,v in pairs:
        if k in out: dup.append(k)
        out[k]=v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out
def load_json(path):
    with open(path,encoding='utf-8') as f:
        try: return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
        except json.JSONDecodeError as e: raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e
def labels(path):
    with open(path,encoding='utf-8') as f: xs=[x.strip() for x in f if x.strip()]
    if not xs: raise ValueError('labels.txt is empty.')
    if len(xs)!=12: raise ValueError(f'Expected 12 labels, found {len(xs)}.')
    if len(set(xs))!=len(xs): raise ValueError('labels.txt contains duplicate labels.')
    return xs
def spans(x, allowed, where):
    if x is None: return set()
    if not isinstance(x,list): raise ValueError(f'{where}: expected a list of entities.')
    out=set()
    for j,s in enumerate(x):
        if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f'{where}, entity {j}: expected [start,end,label] or [start,end,label,entity_text].')
        a,b,l=s[:3]
        if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError(f'{where}, entity {j}: start/end must be integers.')
        if a<0 or b<=a: raise ValueError(f'{where}, entity {j}: require 0 <= start < end.')
        if not isinstance(l,str) or l not in allowed: raise ValueError(f'{where}, entity {j}: invalid label {l!r}.')
        if len(s)==4 and not isinstance(s[3],str): raise ValueError(f'{where}, entity {j}: entity_text must be a string.')
        t=(a,b,l)
        if t in out: raise ValueError(f'{where}, entity {j}: duplicate span {t}.')
        out.add(t)
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 2 entity predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); ap.add_argument('--labels',required=True)
    args=ap.parse_args(); truth=load_json(args.truth); pred=load_json(args.prediction); labs=labels(args.labels)
    if not isinstance(truth,dict) or not truth or not isinstance(pred,dict) or not pred: raise ValueError('Truth and prediction must be non-empty JSON objects.')
    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")
    stats={l:[0,0,0] for l in labs}; allowed=set(labs)
    for did in truth:
        g=spans(truth[did],allowed,f'truth[{did}]'); p=spans(pred[did],allowed,f'prediction[{did}]')
        for l in labs:
            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}
            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)
    precisions=[]; recalls=[]; f1s=[]
    print(f'Examples: {len(truth)}')
    for l in labs:
        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; f=2*pr*rc/(pr+rc) if pr+rc else 0
        precisions.append(pr); recalls.append(rc); f1s.append(f)
        print(f'{l}: P={pr:.6f} R={rc:.6f} F1={f:.6f}')
    print(f'Strict Macro-F1:        {sum(f1s)/len(f1s):.6f}')
    print(f'Strict Macro-Precision: {sum(precisions)/len(precisions):.6f}')
    print(f'Strict Macro-Recall:    {sum(recalls)/len(recalls):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)

In [ ]:
import json, glob, os, subprocess, sys
rows = []
for path in sorted(glob.glob("/kaggle/working/ner_runs/*/summary.json")):
    s = json.load(open(path)); rows.append(s)
    d = os.path.dirname(path)
    if os.path.exists(f"{d}/val_prediction.json"):
        r = subprocess.run([sys.executable, "evaluate_phase2.py", "--truth", f"{d}/val_truth.json",
                            "--prediction", f"{d}/val_prediction.json", "--labels", "/kaggle/working/p2data/labels.txt"],
                           capture_output=True, text=True)
        s["official"] = [l for l in r.stdout.splitlines() if l.startswith("Strict Macro-F1")] or [r.stderr.strip()[-200:]]
print(f"{'experiment':24s} {'arch':9s} {'method':8s} {'train%':>7s} {'val F1':>7s} {'P':>6s} {'R':>6s} {'ep':>3s} {'GPU GB':>6s}  official")
for s in sorted(rows, key=lambda s: -s.get("val_macro_f1", -1)):
    print(f"{s['exp']:24s} {s['arch']:9s} {s['method']:8s} {100*s['params_trainable']/s['params_total']:7.2f} "
          f"{s.get('val_macro_f1', float('nan')):7.4f} {s.get('val_macro_p', float('nan')):6.3f} {s.get('val_macro_r', float('nan')):6.3f} "
          f"{s['best_epoch']:3d} {str(s['peak_gpu_gb']):>6s}  {s.get('official', [''])[0]}")
if rows:
    labels = list(rows[0].get("val_per_label_f1", {}))
    print("\nper-label validation F1:")
    print(f"{'experiment':24s} " + " ".join(f"{l[:6]:>6s}" for l in labels))
    for s in rows:
        if "val_per_label_f1" in s:
            print(f"{s['exp']:24s} " + " ".join(f"{s['val_per_label_f1'][l]:6.3f}" for l in labels))

## N11. Error analysis for one experiment (boundary vs label errors, examples) — material for the report

In [ ]:
import json, glob, os, ner_common as L
EXP = None                     # <- name of an experiment to analyse; None = the best one finished in THIS session
done = {os.path.basename(os.path.dirname(p)): json.load(open(p)).get("val_macro_f1", -1)
        for p in glob.glob("/kaggle/working/ner_runs/*/summary.json")}
done = {k: v for k, v in done.items() if os.path.exists(f"/kaggle/working/ner_runs/{k}/val_prediction.json")}
if EXP not in done:
    EXP = max(done, key=done.get) if done else None
if EXP is None:
    print("No experiment with validation predictions in this session — nothing to analyse.")
else:
    print("analysing:", EXP)
texts, gold, fold, test = L.load_data()
d0 = f"/kaggle/working/ner_runs/{EXP}"
pred = json.load(open(f"{d0}/val_prediction.json")) if EXP else {}
truth = json.load(open(f"{d0}/val_truth.json")) if EXP else {}
if EXP:
    print("error types:", L.error_breakdown(truth, pred))
shown = 0
for d in truth:
    g = {(s, e, l) for s, e, l in truth[d]}; p = {(s, e, l) for s, e, l in pred[d]}
    for s, e, l in sorted(p - g):
        near = [x for x in g if x[0] < e and s < x[1]]
        if near and shown < 15:
            gs, ge, gl = near[0]
            print(f"  pred {l:9s} {texts[d][s:e]!r:45s} | gold {gl:9s} {texts[d][gs:ge]!r}")
            shown += 1

## N12. Check the submission zips before uploading to Codabench

In [ ]:
import zipfile, json, glob
test_ids = set(json.load(open("/kaggle/working/p2data/fold3-text.json")))
labels = {l.strip() for l in open("/kaggle/working/p2data/labels.txt") if l.strip()}
for z in sorted(glob.glob("/kaggle/working/ner_runs/*/submission_*.zip")):
    with zipfile.ZipFile(z) as zf:
        names, pred = zf.namelist(), json.loads(zf.read("prediction.json"))
    n = sum(len(v) for v in pred.values())
    dup = sum(len(v) - len({tuple(x) for x in v}) for v in pred.values())
    bad = sum(1 for v in pred.values() for x in v if x[2] not in labels or not (0 <= x[0] < x[1]))
    ok = names == ["prediction.json"] and set(pred) == test_ids and dup == 0 and bad == 0
    print(f"{z.split('/')[-1]:45s} {'OK' if ok else 'PROBLEM'} | docs {len(pred)} | entities {n} | duplicates {dup} | invalid {bad}")

## N13. Decoding comparison on CPU (no retraining) — uses each experiment's saved validation scores
For every finished experiment: strict macro-F1 on validation with token-level vs piece-level decoding, and
lenient vs strict BIO repair. (A model trained with align="token" can still be DECODED at piece level: each piece
takes the tag of its first token.) Material for the brief's "span-decoding strategies" question.

In [ ]:
import json, glob, os, numpy as np, ner_common as L
texts, gold, fold, test = L.load_data()
C = json.load(open("/kaggle/working/run_config_ner.json"))
tok = L.SplitTokenizer(L.bpe_tok_bpe())
_, val_ids, _ = L.make_split(gold, n_val=C["n_val_docs"], target=C["val_fraction"])
enc = {d: tok.encode_full(texts[d]) for d in val_ids}
lead = {d: L.lead_tokens(enc[d][1], enc[d][2], len(enc[d][3])) for d in val_ids}
G = {d: gold[d] for d in val_ids}
print(f"{'experiment':24s} {'trained':8s} {'token+lenient':>13s} {'token+strict':>12s} {'piece+lenient':>13s} {'piece+strict':>12s}")
for path in sorted(glob.glob("/kaggle/working/ner_runs/*/val_logits.npz")):
    exp = os.path.basename(os.path.dirname(path))
    s = json.load(open(path.replace("val_logits.npz", "summary.json")))
    z = np.load(path); row = []
    for align in ("token", "piece"):
        for mode in ("lenient", "strict"):
            preds = {}
            for d in val_ids:
                lg = z[d].astype(np.float32)
                if align == "piece":
                    keep = [(i, sp) for i, sp in zip(lead[d], enc[d][3]) if i >= 0]
                    preds[d] = L.decode([int(lg[i].argmax()) for i, _ in keep], [sp for _, sp in keep], mode)
                else:
                    preds[d] = L.decode(lg.argmax(-1), enc[d][1], mode)
            row.append(L.strict_scores(G, preds)["macro_f1"])
    print(f"{exp:24s} {s.get('align', 'token'):8s} " + " ".join(f"{v:13.4f}" if i % 2 == 0 else f"{v:12.4f}" for i, v in enumerate(row)))